# Practical P13: Handling API Responses & Status Codes
**Learning Outcome**: Handle any API response status code with appropriate action.

## Part 1: Automatic exception raising with `raise_for_status()`
Using `raise_for_status()` triggers a `requests.exceptions.HTTPError` if the response status is 4xx or 5xx.


In [ ]:
import requests

# Attempt to hit a non-existent URL (404 Error)
bad_url = 'https://httpbin.org/status/404'
try:
    resp = requests.get(bad_url)
    resp.raise_for_status()
except requests.exceptions.HTTPError as err:
    print('Caught expected HTTPError:', err)
    print('Error code was:', err.response.status_code)


## Part 2: Distinguishing Status Codes in Logic
In AI clients, you want different strategies based on failure category:
- **401 Unauthorized**: Stop execution immediately (wrong key).
- **429 Rate Limited**: Wait (back off) and retry later.
- **500 Server Error**: Retry a few times, then report service down.


In [ ]:
def handle_status_code(status):
    # Mock check of status codes
    if status == 200:
        return 'Success'
    elif status == 401:
        return 'Authentication Failure: Invalid API Key'
    elif status == 429:
        return 'Rate Limit Exceeded: Wait and retry'
    elif status >= 500:
        return 'Internal Server Error: AI service currently unavailable'
    else:
        return f'Unhandled status code: {status}'

print(handle_status_code(401))
print(handle_status_code(429))
print(handle_status_code(503))


## Part 3: Using the Timeout parameter
Never make API requests without a timeout config, otherwise your scripts might hang indefinitely if a server becomes unresponsive.
Syntax: `requests.get(url, timeout=5)` (value is in seconds).


In [ ]:
# Test a timeout on a slow connection
delay_url = 'https://httpbin.org/delay/5'
try:
    print('Sending request with 2 second timeout threshold...')
    requests.get(delay_url, timeout=2)
except requests.exceptions.Timeout as e:
    print('Caught expected Timeout exception:', e)


## Hands-On Exercise
**Task**: Write a robust caller function `safe_api_fetch(url, timeout_secs)` that handles:
1. `requests.exceptions.Timeout` -> Print 'Request timed out!'
2. `requests.exceptions.HTTPError` -> Parse response code and print matching message.
3. General `requests.exceptions.RequestException` -> Print 'General network failure.'
Test it against: `https://httpbin.org/status/429`, `https://httpbin.org/status/500`, and `https://httpbin.org/delay/3` (with a timeout of 1).


In [ ]:
# TODO: Write safe_api_fetch logic
def safe_api_fetch(url, timeout_secs=2):
    try:
        r = requests.get(url, timeout=timeout_secs)
        r.raise_for_status()
        print(f'Success for {url}: Status {r.status_code}')
    except requests.exceptions.Timeout:
        print(f'Timed out for {url}')
    except requests.exceptions.HTTPError as e:
        code = e.response.status_code
        if code == 429:
            print(f'HTTP 429 Rate Limited for {url}')
        elif code == 500:
            print(f'HTTP 500 Internal Server Error for {url}')
        else:
            print(f'HTTP Error {code} for {url}')
    except requests.exceptions.RequestException:
        print(f'General error for {url}')

# Testing
safe_api_fetch('https://httpbin.org/status/429')
safe_api_fetch('https://httpbin.org/status/500')
safe_api_fetch('https://httpbin.org/delay/3', timeout_secs=1)
